# FlowSync - Real-World CCTV YOLOv8 Training

This notebook trains the FlowSync vehicle detection model using the `processed` dataset you compiled. It is optimized for Google Colab (using the free NVIDIA T4 GPU).

### Step 1: Mount Google Drive
Run this cell to connect your Google Drive so Colab can access your uploaded dataset and save your trained model so it isn't lost when Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Install Ultralytics (YOLOv8)
Install the official Ultralytics library which contains the YOLOv8 architecture.

In [ ]:
!pip install ultralytics clearml
import ultralytics
ultralytics.checks()

### Step 3: Unzip the Dataset
This unzips the `processed.zip` file you uploaded to your Google Drive into the Colab environment's local storage for ultra-fast training speeds.

In [ ]:
!mkdir -p /content/dataset
!unzip -q "/content/drive/MyDrive/processed.zip" -d /content/dataset
print("Dataset successfully extracted to /content/dataset!")

### Step 4: Fix dataset.yaml paths
We need to update the `dataset.yaml` file so YOLO knows exactly where the unzipped images are located inside Colab.

In [ ]:
import yaml

yaml_path = '/content/dataset/dataset.yaml'
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update path to Colab local directory
data['path'] = '/content/dataset'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, sort_keys=False)

print("dataset.yaml updated successfully!")

### Step 5: Train the Model!
We train the YOLOv8 Nano (`yolov8n.pt`) model for 50 epochs. This will take roughly 2 to 4 hours on the Colab T4 GPU.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 Nano model (best balance of speed/accuracy for CCTV)
model = YOLO('yolov8n.pt')

# Train the model on our custom dataset
results = model.train(
    data='/content/dataset/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0, # Use the NVIDIA GPU
    project='/content/drive/MyDrive/FlowSync_YOLO',
    name='production_run',
    exist_ok=True,
    save=True
)

### Step 6: Export to ONNX
Exporting the model to ONNX format allows it to run 2x-3x faster when you download it back to your Windows PC.

In [ ]:
# Export to ONNX
model.export(format='onnx', opset=12, simplify=True)
print("Model exported to ONNX successfully! You can find it in your Google Drive under FlowSync_YOLO/production_run/weights/")